# LeetCode 3544 — Subtree Inversion Sum bằng Python

- Link: https://leetcode.com/problems/subtree-inversion-sum/
- Độ khó: **Hard**
- Chủ đề: **Quy hoạch động trên cây** với trạng thái `(chẵn lẻ số lần đảo dấu, khoảng cách tới lần đảo gần nhất)`, **cắt trạng thái** `min(d, k)`, cộng mảng theo con

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def subtreeInversionSum(self, edges: List[List[int]], nums: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Hiểu phép "đảo cây con" nhân `−1` cho mọi giá trị trong cây con ⇒ dấu cuối cùng của một đỉnh = `(−1)^{số tổ tiên (kể cả chính nó) bị đảo}`.
2. Thiết kế trạng thái DP `f[u][p][d]`: `p` = chẵn lẻ số lần đảo từ tổ tiên, `d` = khoảng cách tới tổ tiên bị đảo gần nhất (**cắt ở `k`**).
3. Viết chuyển trạng thái "đảo / không đảo" và nhận ra chỉ được đảo khi `d = k`.
4. Cài đặt không đệ quy, tính tổng theo con bằng cộng **mảng độ dài `k`** ⇒ `O(n·k)`.
5. Kiểm chứng bằng vét cạn mọi tập đỉnh bị đảo trên cây nhỏ.

## 2. Đề bài & ví dụ

Cây vô hướng gốc `0`, `n` đỉnh, giá trị `nums[i]`, số nguyên `k`.

- **Đảo cây con** tại đỉnh `x`: mọi giá trị trong cây con gốc `x` nhân `−1`.
- **Ràng buộc khoảng cách:** nếu đảo hai đỉnh `a`, `b` mà một đỉnh là tổ tiên của đỉnh kia thì khoảng cách (số cạnh) giữa chúng phải **≥ k**.

Trả về tổng lớn nhất của các giá trị sau khi đảo (chọn tập đỉnh đảo tùy ý thỏa ràng buộc).

**Ví dụ 1:** `edges=[[0,1],[0,2],[1,3],[1,4],[2,5],[2,6]], nums=[4,-8,-6,3,7,-2,5], k=2` → `27` (đảo 0, 3, 4, 6).

**Ví dụ 2:** `edges=[[0,1],[1,2],[2,3],[3,4]], nums=[-1,3,-2,4,-5], k=2` → `9` (đảo 4).

**Ví dụ 3:** `edges=[[0,1],[0,2]], nums=[0,-1,-2], k=3` → `3` (đảo 1 và 2 — không có quan hệ tổ tiên nên không bị ràng buộc).

**Ràng buộc:** `2 ≤ n ≤ 5·10^4`, `|nums[i]| ≤ 5·10^4`, `1 ≤ k ≤ 50`.

In [1]:
from typing import List
examples = [
    ([[0, 1], [0, 2], [1, 3], [1, 4], [2, 5], [2, 6]], [4, -8, -6, 3, 7, -2, 5], 2, 27),
    ([[0, 1], [1, 2], [2, 3], [3, 4]], [-1, 3, -2, 4, -5], 2, 9),
    ([[0, 1], [0, 2]], [0, -1, -2], 3, 3),
]
for e, nums, k, exp in examples:
    print(e, nums, k, '-> kỳ vọng', exp)

[[0, 1], [0, 2], [1, 3], [1, 4], [2, 5], [2, 6]] [4, -8, -6, 3, 7, -2, 5] 2 -> kỳ vọng 27
[[0, 1], [1, 2], [2, 3], [3, 4]] [-1, 3, -2, 4, -5] 2 -> kỳ vọng 9
[[0, 1], [0, 2]] [0, -1, -2] 3 -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Dựng cây có gốc từ danh sách cạnh vô hướng

BFS từ 0 cho `par` và thứ tự `order` (cha trước con). Đi **ngược** `order` để xử lý con trước cha (thay cho DFS đệ quy).

In [2]:
def rooted(n, edges):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
    par, dep = [-1] * n, [0] * n
    seen = [False] * n
    seen[0] = True
    order = [0]
    for u in order:
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True
                par[v], dep[v] = u, dep[u] + 1
                order.append(v)
    return par, dep, order

print(rooted(7, examples[0][0]))

([-1, 0, 0, 1, 1, 2, 2], [0, 1, 1, 2, 2, 2, 2], [0, 1, 2, 3, 4, 5, 6])


### 3.2. Dấu cuối cùng của một đỉnh

Đỉnh `v` bị nhân `−1` một lần cho **mỗi** tổ tiên (kể cả chính nó) bị đảo. Vậy dấu = `(−1)^{c}` với `c` là số đỉnh bị đảo trên đường từ gốc tới `v`. Chỉ **chẵn lẻ** của `c` là quan trọng.

In [3]:
def final_values(n, edges, nums, inverted):
    par, _, _ = rooted(n, edges)
    out = []
    for v in range(n):
        c, x = 0, v
        while x != -1:
            c += x in inverted
            x = par[x]
        out.append(nums[v] * (-1) ** c)
    return out

vals = final_values(7, examples[0][0], examples[0][1], {0, 3, 4, 6})
print(vals, 'tổng =', sum(vals))

[-4, 8, 6, 3, 7, 2, 5] tổng = 27


### 3.3. Cộng mảng theo phần tử

`[a + b for a, b in zip(A, B)]` — cộng hai mảng cùng độ dài trong một list comprehension (nhanh hơn vòng `for` chỉ số).

In [4]:
A, B = [1, 2, 3], [10, 20, 30]
print([a + b for a, b in zip(A, B)])

[11, 22, 33]


### 3.4. "Cắt" trạng thái khoảng cách

Ràng buộc chỉ quan tâm khoảng cách tới tổ tiên bị đảo **gần nhất** có ≥ `k` hay chưa. Mọi khoảng cách `≥ k` (kể cả "chưa có tổ tiên nào bị đảo") tương đương nhau ⇒ lưu `d = min(khoảng cách, k)`, chỉ có `k` giá trị `1..k`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đứng tại đỉnh `u`, những gì "bên trên" ảnh hưởng tới quyết định trong cây con của `u` chỉ gồm:

1. Dấu hiện tại áp lên cây con (`p` = chẵn lẻ số lần đảo ở tổ tiên).
2. Có được phép đảo `u` không ⇒ khoảng cách tới tổ tiên bị đảo gần nhất (`d`).

Tính tay ví dụ 2 (đường thẳng `0-1-2-3-4`, `nums = [-1,3,-2,4,-5]`, `k = 2`): không đảo gì: tổng `-1`. Đảo 4: `-1+3-2+4+5 = 9`. Đảo 2 (và không gì khác): `-1+3+2-4+5 = 5`. Đảo 2 và 4 (khoảng cách 2 ≥ k): `-1+3+2-4-5 = -5`... Thử nhiều cách, tốt nhất `9`.

In [5]:
from itertools import combinations
e, nums, k, _ = examples[1]
for S in [set(), {4}, {2}, {2, 4}, {0}, {0, 2, 4}, {1, 3}]:
    print(sorted(S), '->', sum(final_values(5, e, nums, S)))

[] -> -1
[4] -> 9
[2] -> 5
[2, 4] -> -5
[0] -> 1
[0, 2, 4] -> 5
[1, 3] -> -3


## 5. Cách 1 — Brute Force: thử mọi tập đỉnh bị đảo

Duyệt `2^n` tập; bỏ tập vi phạm ràng buộc (cặp tổ tiên–hậu duệ có khoảng cách `< k`); tính tổng sau khi đảo.

In [6]:
def inversion_brute(edges, nums, k):
    n = len(nums)
    par, dep, _ = rooted(n, edges)
    anc = []
    for v in range(n):
        lst, x = [], v
        while x != -1:
            lst.append(x)
            x = par[x]
        anc.append(set(lst))
    best = None
    for mask in range(1 << n):
        S = [i for i in range(n) if mask >> i & 1]
        if any(a != b and a in anc[b] and dep[b] - dep[a] < k for a in S for b in S):
            continue
        tot = 0
        for v in range(n):
            c = sum(1 for a in anc[v] if mask >> a & 1)
            tot += -nums[v] if c & 1 else nums[v]
        best = tot if best is None else max(best, tot)
    return best

for e, nums, k, exp in examples:
    print(inversion_brute(e, nums, k), '| kỳ vọng', exp)

27 | kỳ vọng 27
9 | kỳ vọng 9
3 | kỳ vọng 3


**Độ phức tạp:** `O(2^n · n^2)` — chỉ dùng cho `n ≤ 12`.

**Nút thắt:** các lựa chọn ở hai nhánh khác nhau độc lập khi đã biết trạng thái `(p, d)` truyền từ trên xuống ⇒ DP trên cây.

## 6. Ý tưởng thuật toán / bước đột phá

`f[u][p][d]` = tổng lớn nhất của cây con `u`, biết dấu từ tổ tiên là `(−1)^p` và khoảng cách tới tổ tiên bị đảo gần nhất là `d` (đã cắt ở `k`; gốc bắt đầu với `d = k`).

Gọi `s = (−1)^p`, `x = nums[u]`:

- **Không đảo `u`:** `s·x + Σ_con f[c][p][min(d + 1, k)]`.
- **Đảo `u`** (chỉ khi `d = k`): `−s·x + Σ_con f[c][p ⊕ 1][1]`.

`f[u][p][d]` = max của các lựa chọn hợp lệ. Đáp án `f[0][0][k]`.

Quy tắc:

- **Nếu** `d < k` **thì** không được đảo `u` (quá gần lần đảo trước).
- **Nếu** đảo `u` **thì** các con bắt đầu đếm lại từ `d = 1` và dấu bị lật.
- **Nếu** không đảo **thì** các con nhận `d + 1` (cắt ở `k`).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ (top-down)

Viết trực tiếp công thức bằng `lru_cache`. Dễ hiểu nhưng đệ quy sâu tới `n` và có `2·n·k` trạng thái gọi hàm — chỉ phù hợp cây nhỏ/vừa.

In [7]:
import sys
from functools import lru_cache
sys.setrecursionlimit(10000)

def inversion_memo(edges, nums, k):
    n = len(nums)
    par, _, order = rooted(n, edges)
    children = [[] for _ in range(n)]
    for v in order[1:]:
        children[par[v]].append(v)

    @lru_cache(maxsize=None)
    def f(u, p, d):
        s = -1 if p else 1
        best = s * nums[u] + sum(f(c, p, min(d + 1, k)) for c in children[u])
        if d == k:
            best = max(best, -s * nums[u] + sum(f(c, p ^ 1, 1) for c in children[u]))
        return best

    return f(0, 0, k)

for e, nums, k, exp in examples:
    print(inversion_memo(e, nums, k), '| kỳ vọng', exp)

27 | kỳ vọng 27
9 | kỳ vọng 9
3 | kỳ vọng 3


### 7.2. Cách 3 — Bottom-up với bảng `f[u][p][d]` (vòng lặp từng ô)

In [8]:
def inversion_table(edges, nums, k):
    n = len(nums)
    par, _, order = rooted(n, edges)
    children = [[] for _ in range(n)]
    for v in order[1:]:
        children[par[v]].append(v)
    f = [None] * n                       # f[u] = [[...k phần tử cho p=0], [... cho p=1]]
    for u in reversed(order):
        res = [[0] * (k + 1) for _ in range(2)]     # chỉ số d = 1..k
        for p in (0, 1):
            s = -1 if p else 1
            for d in range(1, k + 1):
                nd = min(d + 1, k)
                best = s * nums[u] + sum(f[c][p][nd] for c in children[u])
                if d == k:
                    best = max(best, -s * nums[u] + sum(f[c][p ^ 1][1] for c in children[u]))
                res[p][d] = best
        f[u] = res
    return f[0][0][k]

for e, nums, k, exp in examples:
    print(inversion_table(e, nums, k), '| kỳ vọng', exp)

27 | kỳ vọng 27
9 | kỳ vọng 9
3 | kỳ vọng 3


**Độ phức tạp:** mỗi cạnh cha–con được dùng `2k` lần ⇒ `O(n·k)` phép cộng, nhưng vòng lặp Python lồng nhau nên hằng số lớn.

### 7.3. Cách 4 — Cộng mảng con dồn lên cha (lời giải chính)

Gọi `C[u][p]` = mảng `Σ_con f[c][p][·]` (độ dài `k`, chỉ số `e − 1` cho `e = 1..k`). Khi đó:

- `f[u][p][d] = s·x + C[u][p][min(d+1,k) − 1]` ⇒ cả mảng là `C[p][1:] + [C[p][k−1]]` cộng `s·x`.
- Phần tử cuối (`d = k`) lấy `max` với `−s·x + C[u][p⊕1][0]`.

Xử lý xong `u` thì **cộng mảng** `f[u][p]` vào `C[par[u]][p]`.

In [9]:
def inversion_arrays(edges, nums, k):
    n = len(nums)
    par, _, order = rooted(n, edges)
    C0, C1 = [None] * n, [None] * n
    zero = [0] * k
    for u in reversed(order):
        c0, c1 = C0[u] or zero, C1[u] or zero
        x = nums[u]
        f0 = [x + v for v in c0[1:]] + [max(x + c0[k - 1], -x + c1[0])]
        f1 = [-x + v for v in c1[1:]] + [max(-x + c1[k - 1], x + c0[0])]
        p = par[u]
        if p < 0:
            return f0[k - 1]
        if C0[p] is None:
            C0[p], C1[p] = f0, f1
        else:
            C0[p] = [a + b for a, b in zip(C0[p], f0)]
            C1[p] = [a + b for a, b in zip(C1[p], f1)]

for e, nums, k, exp in examples:
    print(inversion_arrays(e, nums, k), '| kỳ vọng', exp)

27 | kỳ vọng 27
9 | kỳ vọng 9
3 | kỳ vọng 3


Với `k = 1`: `c0[1:]` rỗng, mảng chỉ có một phần tử `d = 1 = k` — luôn được đảo, đúng như ràng buộc (khoảng cách ≥ 1 luôn thỏa).

**Độ phức tạp:** `O(n·k)` thời gian, `O(n·k)` bộ nhớ trường hợp xấu (thực tế giải phóng dần). Với `n = 5·10^4`, `k = 50`: ~`5·10^6` phép cộng trong list comprehension ⇒ < 1 giây.

## 8. TRACE từng bước

Trace cách 4 trên ví dụ 1 (`k = 2`): in `f[u][0]` và `f[u][1]` (chỉ số `d = 1, 2`) cho từng đỉnh từ lá lên, kèm giải thích lựa chọn ở `d = k`.

In [10]:
def trace(edges, nums, k):
    n = len(nums)
    par, _, order = rooted(n, edges)
    C0, C1 = [None] * n, [None] * n
    zero = [0] * k
    for u in reversed(order):
        c0, c1 = C0[u] or zero, C1[u] or zero
        x = nums[u]
        keep0, flip0 = x + c0[k - 1], -x + c1[0]
        f0 = [x + v for v in c0[1:]] + [max(keep0, flip0)]
        f1 = [-x + v for v in c1[1:]] + [max(-x + c1[k - 1], x + c0[0])]
        print(f"đỉnh {u} (x={x:>2}): tổng con p=0 {c0}, p=1 {c1} | f[p=0] = {f0}, f[p=1] = {f1}"
              f" | tại d=k, p=0: giữ {keep0} vs đảo {flip0} -> {'ĐẢO' if flip0 > keep0 else 'giữ'}")
        p = par[u]
        if p < 0:
            print('Đáp án =', f0[k - 1])
            return f0[k - 1]
        C0[p] = f0 if C0[p] is None else [a + b for a, b in zip(C0[p], f0)]
        C1[p] = f1 if C1[p] is None else [a + b for a, b in zip(C1[p], f1)]

trace(*examples[0][:3])

đỉnh 6 (x= 5): tổng con p=0 [0, 0], p=1 [0, 0] | f[p=0] = [5, 5], f[p=1] = [-5, 5] | tại d=k, p=0: giữ 5 vs đảo -5 -> giữ
đỉnh 5 (x=-2): tổng con p=0 [0, 0], p=1 [0, 0] | f[p=0] = [-2, 2], f[p=1] = [2, 2] | tại d=k, p=0: giữ -2 vs đảo 2 -> ĐẢO
đỉnh 4 (x= 7): tổng con p=0 [0, 0], p=1 [0, 0] | f[p=0] = [7, 7], f[p=1] = [-7, 7] | tại d=k, p=0: giữ 7 vs đảo -7 -> giữ
đỉnh 3 (x= 3): tổng con p=0 [0, 0], p=1 [0, 0] | f[p=0] = [3, 3], f[p=1] = [-3, 3] | tại d=k, p=0: giữ 3 vs đảo -3 -> giữ
đỉnh 2 (x=-6): tổng con p=0 [3, 7], p=1 [-3, 7] | f[p=0] = [1, 3], f[p=1] = [13, 13] | tại d=k, p=0: giữ 1 vs đảo 3 -> ĐẢO
đỉnh 1 (x=-8): tổng con p=0 [10, 10], p=1 [-10, 10] | f[p=0] = [2, 2], f[p=1] = [18, 18] | tại d=k, p=0: giữ 2 vs đảo -2 -> giữ
đỉnh 0 (x= 4): tổng con p=0 [3, 5], p=1 [31, 31] | f[p=0] = [9, 27], f[p=1] = [27, 27] | tại d=k, p=0: giữ 9 vs đảo 27 -> ĐẢO
Đáp án = 27


27

Đọc trace: ở gốc 0 với `p = 0, d = k`, phương án "đảo" thắng (đúng như đề: đảo 0); các lá 3, 4, 6 sau đó được đảo lại với trạng thái `p = 1`.

## 9. Code nộp LeetCode

In [11]:
class Solution:
    def subtreeInversionSum(self, edges: List[List[int]], nums: List[int], k: int) -> int:
        n = len(nums)
        adj = [[] for _ in range(n)]
        for a, b in edges:
            adj[a].append(b)
            adj[b].append(a)
        par = [-1] * n
        seen = [False] * n
        seen[0] = True
        order = [0]
        for u in order:
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    par[v] = u
                    order.append(v)
        C0, C1 = [None] * n, [None] * n
        zero = [0] * k
        for u in reversed(order):
            c0 = C0[u] or zero
            c1 = C1[u] or zero
            x = nums[u]
            f0 = [x + v for v in c0[1:]]
            f0.append(max(x + c0[k - 1], -x + c1[0]))
            f1 = [-x + v for v in c1[1:]]
            f1.append(max(-x + c1[k - 1], x + c0[0]))
            p = par[u]
            if p < 0:
                return f0[k - 1]
            if C0[p] is None:
                C0[p], C1[p] = f0, f1
            else:
                C0[p] = [a + b for a, b in zip(C0[p], f0)]
                C1[p] = [a + b for a, b in zip(C1[p], f1)]
            C0[u] = C1[u] = None
        return 0

for e, nums, k, exp in examples:
    got = Solution().subtreeInversionSum(e, nums, k)
    print(got, 'OK' if got == exp else 'SAI')

27 OK
9 OK
3 OK


## 10. Vì sao thuật toán đúng

1. **Trạng thái đủ:** quyết định trong cây con `u` chỉ phụ thuộc (a) dấu đang áp lên cây con — xác định bởi chẵn lẻ số lần đảo ở tổ tiên; (b) có được đảo các đỉnh trong cây con hay không — ràng buộc chỉ liên quan **tổ tiên bị đảo gần nhất** (mọi tổ tiên bị đảo xa hơn còn xa hơn nữa), và chỉ cần biết khoảng cách có `≥ k` hay không cho mỗi độ sâu ⇒ `d = min(khoảng cách, k)` là đủ.
2. **Độc lập giữa các con:** với `(p, d)` cố định truyền xuống, ràng buộc giữa hai đỉnh thuộc hai cây con khác nhau không tồn tại (không phải tổ tiên–hậu duệ) ⇒ cộng tối ưu của từng con.
3. **Chuyển trạng thái đúng:** không đảo `u`: giá trị `u` mang dấu `s`, con cách tổ tiên bị đảo gần nhất thêm 1 cạnh. Đảo `u`: giá trị mang dấu `−s`, `u` trở thành tổ tiên bị đảo gần nhất của con (khoảng cách 1), dấu cây con lật.
4. **Gốc:** chưa có tổ tiên nào ⇒ tương đương khoảng cách vô hạn ⇒ `d = k`, `p = 0`.

## 11. Phân tích Time / Space Complexity

Gọi `n ≤ 5·10^4`, `k ≤ 50`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^n n^2)` | `O(n)` |
| 2. Đệ quy có nhớ | `O(n·k)` trạng thái × số con | `O(n·k)` + stack đệ quy |
| 3. Bảng từng ô | `O(n·k)` (hằng số lớn) | `O(n·k)` |
| 4. Cộng mảng dồn lên cha | `O(n·k)` | `O(n·k)` xấu nhất |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {
    'memo': inversion_memo,
    'table': inversion_table,
    'arrays': inversion_arrays,
    'submit': lambda e, nums, k: Solution().subtreeInversionSum(e, nums, k),
}
edge = [
    ([[0, 1]], [-5, -7], 1, 12),            # k = 1: đảo 0 (lật cả hai) -> 12
    ([[0, 1]], [-5, 7], 1, 12),             # đảo 0 rồi đảo 1 (khoảng cách 1 >= k)
    ([[0, 1]], [-5, 7], 2, 2),              # chỉ đảo một trong hai: đảo 0 -> 5 - 7 = -2; không đảo -> 2
    ([[0, 1], [1, 2]], [1, 2, 3], 50, 6),   # toàn dương: không đảo
    ([[0, 1], [0, 2], [0, 3]], [0, -1, -1, -1], 2, 3),
]
for e, nums, k, exp in examples + edge:
    for name, f in impls.items():
        got = f(e, nums, k)
        assert got == exp, (name, e, nums, k, got, exp)
    assert inversion_brute(e, nums, k) == exp
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [13]:
import random
random.seed(3544)

def rand_tree(n):
    perm = list(range(n))
    random.shuffle(perm)
    perm[perm.index(0)], perm[0] = perm[0], 0
    es = []
    for i in range(1, n):
        a, b = perm[random.randrange(i)], perm[i]
        es.append([a, b] if random.random() < 0.5 else [b, a])
    return es

cnt = 0
for _ in range(400):
    n = random.randint(2, 11)
    e = rand_tree(n)
    nums = [random.randint(-9, 9) for _ in range(n)]
    k = random.randint(1, 5)
    exp = inversion_brute(e, nums, k)
    for name, f in impls.items():
        assert f(e, nums, k) == exp, (name, e, nums, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'cây ngẫu nhiên.')

Khớp vét cạn trên 400 cây ngẫu nhiên.


### 12.3. Ca lớn (`n = 5·10^4`, `k = 50`)

In [14]:
import time
random.seed(50)
n = 5 * 10**4
nums = [random.randint(-5 * 10**4, 5 * 10**4) for _ in range(n)]
for name, e in [('ngẫu nhiên', rand_tree(n)), ('đường thẳng', [[i - 1, i] for i in range(1, n)]),
                ('sao', [[0, i] for i in range(1, n)])]:
    for k in (1, 50):
        t = time.time(); a = Solution().subtreeInversionSum(e, nums, k); t1 = time.time() - t
        print(f"{name:>11}, k={k:>2}: đáp án {a} | {t1:.2f}s")

 ngẫu nhiên, k= 1: đáp án 1243228949 | 0.14s


 ngẫu nhiên, k=50: đáp án 866726631 | 0.46s
đường thẳng, k= 1: đáp án 1243228949 | 0.06s


đường thẳng, k=50: đáp án 270803253 | 0.27s
        sao, k= 1: đáp án 1243228949 | 0.06s


        sao, k=50: đáp án 1243228949 | 0.54s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Áp ràng buộc khoảng cách cho **mọi** cặp đỉnh bị đảo (kể cả không phải tổ tiên–hậu duệ)

In [15]:
def wrong_all_pairs(edges, nums, k):
    n = len(nums)
    par, dep, _ = rooted(n, edges)
    anc = []
    for v in range(n):
        s, x = {}, v
        d = 0
        while x != -1:
            s[x] = d; x = par[x]; d += 1
        anc.append(s)

    def dist(a, b):                      # khoảng cách trên cây qua tổ tiên chung
        return min(anc[a][x] + anc[b][x] for x in anc[a] if x in anc[b])

    best = None
    for mask in range(1 << n):
        S = [i for i in range(n) if mask >> i & 1]
        if any(dist(a, b) < k for a in S for b in S if a < b):      # SAI: áp cho mọi cặp
            continue
        tot = sum(-v if sum(1 for x in anc[i] if mask >> x & 1) & 1 else v for i, v in enumerate(nums))
        best = tot if best is None else max(best, tot)
    return best

random.seed(1)
found = None
for _ in range(3000):
    n = random.randint(3, 7)
    e = [[random.randrange(i), i] for i in range(1, n)]
    nums = [random.randint(-9, 9) for _ in range(n)]
    k = random.randint(2, 4)
    if wrong_all_pairs(e, nums, k) != inversion_brute(e, nums, k):
        found = (e, nums, k)
        break
print('phản ví dụ (edges, nums, k):', found)
print('sai :', wrong_all_pairs(*found), '| đúng:', Solution().subtreeInversionSum(*found))

phản ví dụ (edges, nums, k): ([[0, 1], [1, 2], [2, 3], [2, 4], [3, 5], [5, 6]], [1, 6, 6, -6, -9, 0, 3], 3)
sai : 19 | đúng: 25


**Vì sao sai:** ràng buộc chỉ áp cho cặp **tổ tiên–hậu duệ**. Hai đỉnh ở hai nhánh khác nhau (ví dụ hai anh em, khoảng cách 2) vẫn được đảo cùng lúc dù `k` lớn — như ví dụ 3 của đề (đảo cả 1 và 2 với `k = 3`).

### Lỗi 2 — Không cắt `d` ở `k` mà cho `d` tăng mãi (hoặc dùng mảng kích thước `n`)

In [16]:
def states_without_cap(n, k):
    return 2 * n * n          # d có thể tới n

def states_with_cap(n, k):
    return 2 * n * k

print('không cắt:', states_without_cap(5 * 10**4, 50), 'trạng thái | có cắt:', states_with_cap(5 * 10**4, 50))

không cắt: 5000000000 trạng thái | có cắt: 5000000


**Vì sao sai:** đúng về mặt logic nhưng số trạng thái `O(n^2)` (cây đường thẳng) ⇒ quá chậm/tốn bộ nhớ. Mọi `d ≥ k` tương đương nhau.

### Lỗi 3 — Sau khi đảo `u`, các con vẫn nhận `d + 1` thay vì đặt lại `d = 1`

In [17]:
def wrong_no_reset(edges, nums, k):
    n = len(nums)
    par, _, order = rooted(n, edges)
    C0, C1 = [None] * n, [None] * n
    zero = [0] * k
    for u in reversed(order):
        c0, c1 = C0[u] or zero, C1[u] or zero
        x = nums[u]
        f0 = [x + v for v in c0[1:]] + [max(x + c0[k - 1], -x + c1[k - 1])]   # SAI: c1[k-1] thay vì c1[0]
        f1 = [-x + v for v in c1[1:]] + [max(-x + c1[k - 1], x + c0[k - 1])]
        p = par[u]
        if p < 0:
            return f0[k - 1]
        C0[p] = f0 if C0[p] is None else [a + b for a, b in zip(C0[p], f0)]
        C1[p] = f1 if C1[p] is None else [a + b for a, b in zip(C1[p], f1)]

e, nums, k, exp = examples[1]
print('sai :', wrong_no_reset(e, nums, k), '| đúng:', Solution().subtreeInversionSum(e, nums, k))

sai : 15 | đúng: 9


**Vì sao sai:** cho phép đảo con ngay sát đỉnh vừa đảo ⇒ vi phạm ràng buộc và cho tổng lớn hơn thực tế.

### Lỗi 4 — Quên lật dấu `p` cho cây con khi đảo `u`

In [18]:
def wrong_no_flip(edges, nums, k):
    n = len(nums)
    par, _, order = rooted(n, edges)
    C0, C1 = [None] * n, [None] * n
    zero = [0] * k
    for u in reversed(order):
        c0, c1 = C0[u] or zero, C1[u] or zero
        x = nums[u]
        f0 = [x + v for v in c0[1:]] + [max(x + c0[k - 1], -x + c0[0])]       # SAI: dùng c0 (không lật)
        f1 = [-x + v for v in c1[1:]] + [max(-x + c1[k - 1], x + c1[0])]
        p = par[u]
        if p < 0:
            return f0[k - 1]
        C0[p] = f0 if C0[p] is None else [a + b for a, b in zip(C0[p], f0)]
        C1[p] = f1 if C1[p] is None else [a + b for a, b in zip(C1[p], f1)]

e, nums, k, exp = examples[0]
print('sai :', wrong_no_flip(e, nums, k), '| đúng:', Solution().subtreeInversionSum(e, nums, k))

sai : 31 | đúng: 27


**Vì sao sai:** đảo `u` nhân `−1` **cả cây con**, không chỉ riêng `u`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Thao tác trên cây con, hiệu ứng tích lũy theo tổ tiên" ⇒ trạng thái DP mang **thông tin từ tổ tiên** (chẵn lẻ, khoảng cách…) truyền xuống.
- "Hai thao tác phải cách nhau ít nhất `k`" ⇒ lưu khoảng cách tới thao tác gần nhất, **cắt ở `k`**.
- Lấy tổng theo con của mảng trạng thái ⇒ cộng mảng dồn lên cha theo thứ tự BFS ngược.
- `n` lớn ⇒ không đệ quy.

## 15. Biến thể của bài

1. Ràng buộc áp cho **mọi cặp** đỉnh bị đảo (không chỉ tổ tiên–hậu duệ) ⇒ khó hơn nhiều (giống bài tập độc lập khoảng cách `k` trên cây).
2. Giới hạn **số lần đảo** tối đa `t` ⇒ thêm chiều đếm ⇒ knapsack trên cây.
3. Thao tác là **cộng hằng số** vào cây con thay vì đổi dấu ⇒ trạng thái là tổng tích lũy (không còn hữu hạn) — đổi cách tiếp cận.
4. `k = 1` ⇒ mọi tập đỉnh đều hợp lệ ⇒ có lời giải tham lam: mỗi đỉnh tự chọn dấu tốt nhất cho... (thực tế mỗi đỉnh chọn dấu riêng được, đáp án `Σ |nums|`).

In [19]:
# Biến thể 4: k = 1 -> đáp án = Σ |nums| (mỗi đỉnh có thể tự quyết dấu của nó)
random.seed(4)
for _ in range(200):
    n = random.randint(2, 30)
    e = rand_tree(n)
    nums = [random.randint(-9, 9) for _ in range(n)]
    assert Solution().subtreeInversionSum(e, nums, 1) == sum(abs(v) for v in nums)
print('k = 1: đáp án luôn bằng Σ|nums| (đã kiểm chứng ngẫu nhiên).')

k = 1: đáp án luôn bằng Σ|nums| (đã kiểm chứng ngẫu nhiên).


Vì sao? Với `k = 1`, đặt `t[v] = 1` nếu muốn `v` đổi dấu; đảo đỉnh `v` ⇔ `t[v] ≠ t[cha(v)]` — luôn chọn được.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Vét cạn tập đảo | `O(2^n n^2)` | `O(n)` | Oracle |
| 2 | Đệ quy có nhớ | `O(n k)` | `O(n k)` + stack | Cây nhỏ, dễ viết |
| 3 | Bảng từng ô | `O(n k)` | `O(n k)` | Hiểu chuyển trạng thái |
| 4 | Cộng mảng dồn lên cha | `O(n k)` | `O(n k)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** đường `0-1-2`, `nums = [-3, -3, -3]`, `k = 3`.

**Bài 2 (tính tay):** sao 4 đỉnh, gốc `0` giá trị `5`, ba lá giá trị `-2, -2, 1`, `k = 2`.

**Bài 3 (điền TODO):** viết `transition(x, c0, c1, k)` trả về `(f0, f1)`.

**Bài 4 (tự viết lại):** truy vết **tập đỉnh bị đảo** đạt tối ưu và kiểm tra lại bằng `final_values`.

**Bài 5 (mini challenge):** biến thể 2 — tối đa `t` lần đảo; DP thêm chiều số lần, kiểm chứng bằng vét cạn.

In [20]:
def transition_todo(x, c0, c1, k):
    # TODO
    raise NotImplementedError

def inverted_set_todo(edges, nums, k):
    # TODO
    raise NotImplementedError

def limited_inversions_todo(edges, nums, k, t):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Chỉ đảo được một đỉnh (mọi cặp tổ tiên cách nhau < 3). (2) Đảo 0 lật cả ba. (3) `9`.

**Bài 2.** (1) Đảo gốc: `-5 + 2 + 2 - 1 = -2`, rồi lá không đảo được (khoảng cách 1 < 2). (2) Không đảo gốc, đảo hai lá `-2`: `5 + 2 + 2 + 1 = 10`. (3) `10`.

**Bài 3.** (1) Copy hai dòng `f0`, `f1` của lời giải. (2) Chú ý trường hợp `k = 1`. (3) So với `inversion_table` trên dữ liệu nhỏ.

**Bài 4.** (1) Lưu lựa chọn tại `(u, p, d)`. (2) Đi xuống từ `(0, 0, k)`. (3) Tính tổng bằng `final_values` và so với đáp án.

**Bài 5.** (1) Trạng thái thêm `j` = số lần đã dùng trong cây con. (2) Gộp con bằng knapsack trên `j`. (3) Vét cạn giới hạn `|S| ≤ t`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
a1 = Solution().subtreeInversionSum([[0, 1], [1, 2]], [-3, -3, -3], 3)
a2 = Solution().subtreeInversionSum([[0, 1], [0, 2], [0, 3]], [5, -2, -2, 1], 2)
assert a1 == 9 == inversion_brute([[0, 1], [1, 2]], [-3, -3, -3], 3)
assert a2 == 10 == inversion_brute([[0, 1], [0, 2], [0, 3]], [5, -2, -2, 1], 2)
print('Bài 1:', a1, '| Bài 2:', a2)

Bài 1: 9 | Bài 2: 10


In [22]:
# Bài 3
def transition(x, c0, c1, k):
    f0 = [x + v for v in c0[1:]] + [max(x + c0[k - 1], -x + c1[0])]
    f1 = [-x + v for v in c1[1:]] + [max(-x + c1[k - 1], x + c0[0])]
    return f0, f1

print(transition(4, [0, 0], [0, 0], 2), transition(-3, [5], [-5], 1))

([4, 4], [-4, 4]) ([2], [2])


In [23]:
# Bài 4: truy vết tập đỉnh bị đảo
def inverted_set(edges, nums, k):
    n = len(nums)
    par, _, order = rooted(n, edges)
    children = [[] for _ in range(n)]
    for v in order[1:]:
        children[par[v]].append(v)
    f = [None] * n
    for u in reversed(order):
        c = [[sum(f[ch][p][j] for ch in children[u]) for j in range(k)] for p in (0, 1)]
        f[u] = list(transition(nums[u], c[0], c[1], k))
    S = []
    stack = [(0, 0, k)]
    while stack:
        u, p, d = stack.pop()
        s = -1 if p else 1
        keep = s * nums[u] + sum(f[ch][p][min(d + 1, k) - 1] for ch in children[u])
        flip = -s * nums[u] + sum(f[ch][p ^ 1][0] for ch in children[u]) if d == k else None
        if flip is not None and flip > keep:
            S.append(u)
            stack.extend((ch, p ^ 1, 1) for ch in children[u])
        else:
            stack.extend((ch, p, min(d + 1, k)) for ch in children[u])
    return f[0][0][k - 1], sorted(S)

random.seed(6)
for _ in range(200):
    n = random.randint(2, 10)
    e = rand_tree(n)
    nums = [random.randint(-9, 9) for _ in range(n)]
    k = random.randint(1, 4)
    best, S = inverted_set(e, nums, k)
    assert best == sum(final_values(n, e, nums, set(S))) == inversion_brute(e, nums, k)
print('Bài 4: ví dụ 1 ->', inverted_set(*examples[0][:3]))

Bài 4: ví dụ 1 -> (27, [0, 3, 4, 6])


In [24]:
# Bài 5: tối đa t lần đảo
def limited_inversions(edges, nums, k, t):
    n = len(nums)
    par, _, order = rooted(n, edges)
    children = [[] for _ in range(n)]
    for v in order[1:]:
        children[par[v]].append(v)
    NEG = float('-inf')

    def merge(A, B):
        out = [NEG] * (t + 1)
        for i, a in enumerate(A):
            if a == NEG:
                continue
            for j in range(t + 1 - i):
                if B[j] != NEG and a + B[j] > out[i + j]:
                    out[i + j] = a + B[j]
        return out

    g = [None] * n          # g[u][p][d] = mảng theo số lần dùng
    for u in reversed(order):
        res = [[None] * (k + 1) for _ in range(2)]
        for p in (0, 1):
            s = -1 if p else 1
            for d in range(1, k + 1):
                acc = [0] + [NEG] * t
                for ch in children[u]:
                    acc = merge(acc, g[ch][p][min(d + 1, k)])
                best = [a + s * nums[u] if a != NEG else NEG for a in acc]
                if d == k:
                    acc2 = [0] + [NEG] * t
                    for ch in children[u]:
                        acc2 = merge(acc2, g[ch][p ^ 1][1])
                    for j in range(t):
                        if acc2[j] != NEG:
                            best[j + 1] = max(best[j + 1], acc2[j] - s * nums[u])
                res[p][d] = best
        g[u] = res
    return max(g[0][0][k])

def limited_brute(edges, nums, k, t):
    n = len(nums)
    par, dep, _ = rooted(n, edges)
    anc = []
    for v in range(n):
        s, x = set(), v
        while x != -1:
            s.add(x); x = par[x]
        anc.append(s)
    best = None
    for mask in range(1 << n):
        S = [i for i in range(n) if mask >> i & 1]
        if len(S) > t or any(a != b and a in anc[b] and dep[b] - dep[a] < k for a in S for b in S):
            continue
        tot = sum(-v if sum(1 for a in anc[i] if mask >> a & 1) & 1 else v for i, v in enumerate(nums))
        best = tot if best is None else max(best, tot)
    return best

random.seed(7)
for _ in range(150):
    n = random.randint(2, 8)
    e = rand_tree(n)
    nums = [random.randint(-9, 9) for _ in range(n)]
    k, t = random.randint(1, 3), random.randint(0, 3)
    assert limited_inversions(e, nums, k, t) == limited_brute(e, nums, k, t)
print('Bài 5: khớp vét cạn; ví dụ 1 với t = 1, 2, 4:',
      [limited_inversions(*examples[0][:3], t) for t in (1, 2, 4)])

Bài 5: khớp vét cạn; ví dụ 1 với t = 1, 2, 4: [9, 11, 27]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
f[u][p][d], d = 1..k (cắt ở k), gốc: p = 0, d = k
C[p] = Σ_con f[c][p]  (mảng độ dài k)
f[u][p] = [s*x + C[p][j] for j in 1..k-1] + [max(s*x + C[p][k-1], -s*x + C[p^1][0])]
đáp án = f[0][0][k-1]; cộng f[u][p] vào C[cha][p] theo BFS ngược
```

**Checklist:**

- [ ] Ràng buộc chỉ áp cho cặp **tổ tiên–hậu duệ**?
- [ ] `d` cắt ở `k`; chỉ được đảo khi `d = k`?
- [ ] Đảo `u` ⇒ con nhận `d = 1` và dấu lật?
- [ ] Trường hợp `k = 1`?
- [ ] Không đệ quy, cộng mảng bằng list comprehension?